# 初始化大模型

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv()
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key=os.getenv("DASHSCOPE_API_KEY")
llm=init_chat_model(
    model="qwen3.6-plus",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
    timeout=60,
)

## 联网搜索工具

In [4]:
from langchain_tavily import TavilySearch
from langchain.tools import tool
web_search=TavilySearch(
    max_results=5,
    topic="general"
)
# @ tool
# def get_tavily_search(query: str) -> str:
#     """Get search results from Tavily."""
#     return web_search.invoke(query)

## 数据库会话记忆存储

In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
connection=sqlite3.connect("memory.db", check_same_thread=False)
checkpointer=SqliteSaver(connection)
checkpointer.setup()

## 创建智能体

In [6]:
system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

In [7]:
from langchain.agents import create_agent
agent=create_agent(
    model=llm,
    tools=[web_search],
    checkpointer=checkpointer,
    system_prompt=system_prompt
)

# 在线图片识别

In [9]:
from langchain_core.messages import HumanMessage
message=HumanMessage(
    [
        {"type": "text", "text": "帮我看看这些食材能做什么？"},
        {"type": "image", "url": "https://qcloud.dpfile.com/pc/9JCz9ek0wy_f69WYoPAgaqIhoYWAlUgtga_LwFc2HMO-UhjvYTJFerXjRcdhy0xd.jpg"},
    ]
)
config={"configurable": {"thread_id": "cooking_thread"}}
response=agent.invoke(
    {"messages":[message]},
    config
)

In [11]:
print(response["messages"][-1].content)

### 基于冰箱食材的创意食谱建议报告  

以下结合您冰箱内的**酸奶、生菜、青椒、桃子、蓝莓、柚子、火龙果、葱、西兰花、芦笋、玉米**，从营养与操作难度维度筛选出5款高适配食谱，供您快速决策：  


#### 1. 「彩虹果蔬沙拉」  
- **综合得分**：7.5/10（营养9分 + 难度1.5分）  
- **推荐理由**：  
  - 营养：生菜、青椒、玉米提供膳食纤维与维生素；蓝莓、桃子、火龙果补充花青素、果胶与天然糖分，酸奶替代沙拉酱增加优质蛋白。  
  - 难度：无需烹饪，食材切块/撕碎后混合，淋酸奶即可完成，10分钟搞定。  
- **参考图片**：[Reddit食谱示例](https://www.reddit.com/r/Cooking/comments/cjj8qs/if_you_tell_me_whats_in_your_fridge_or_pantry_ill/?tl=zh-hans)  


#### 2. 「酸奶水果杯」  
- **综合得分**：8/10（营养8.5分 + 难度1分）  
- **推荐理由**：  
  - 营养：酸奶（蛋白、益生菌）搭配桃子、蓝莓、火龙果（维生素C、抗氧化物质），适合作为早餐或下午茶，饱腹且低卡。  
  - 难度：将水果切丁，分层铺入酸奶中，无需加热，3分钟完成。  
- **参考图片**：[知乎酸奶花式做法](https://zhuanlan.zhihu.com/p/28210671)  


#### 3. 「烤时蔬串」  
- **综合得分**：6.5/10（营养8分 + 难度3分）  
- **推荐理由**：  
  - 营养：西兰花、芦笋、玉米、青椒组合覆盖多种维生素（如维生素C、K）与矿物质，烤制保留风味且减少油脂摄入。  
  - 难度：蔬菜切块串签，刷少量橄榄油+盐，烤箱/空气炸锅200℃烤15分钟（新手易掌握火候）。  
- **参考图片**：[Pennsylvania健康饮食指南（烤蔬菜示例）](https://www.pa.gov/content/dam/copapwp-pagov/en/aging/documents/resources-for-aging-professionals/nutrition-facts-sheets/chinese-cantonese/20%